# The existing PDB loader carries no bond orders and no formal charges

`mb.load` reads a PDB file through mdtraj. `Protein` reads the same file
through the residue definitions. Same atoms, same bonds, different chemistry.

In [1]:
import logging, warnings
from rdkit import RDLogger

warnings.filterwarnings("ignore")
logging.disable(logging.WARNING)
RDLogger.DisableLog("rdApp.*")

In [2]:
from collections import Counter

import mbuild as mb
from mbuild.biopolymers import Protein

generic = mb.load("../semaglutide_apo.pdb")
protein = Protein("../semaglutide_apo.pdb")


def bond_orders(compound):
    return dict(Counter(d["bond_order"] for *_, d in compound.bonds(return_bond_order=True)))


print("mb.load :", generic.n_particles, "atoms", generic.n_bonds, "bonds", bond_orders(generic))
print("Protein :", protein.n_particles, "atoms", protein.n_bonds, "bonds", bond_orders(protein))

mb.load : 470 atoms 475 bonds {0.0: 475}
Protein : 470 atoms 475 bonds {1.0: 422, 2.0: 53}


In [3]:
print("mb.load particle charges:", dict(Counter(p.charge for p in generic.particles())))
print("Protein net formal charge:", protein.net_formal_charge)
print({f"{r.name}{r.resnum}": r.formal_charge for r in protein.residues() if r.formal_charge})

mb.load particle charges: {None: 470}
Protein net formal charge: -1
{'HIS1': 1, 'GLU3': -1, 'ASP9': -1, 'GLU15': -1, 'LYS20': 1, 'GLU21': -1, 'ARG28': 1, 'ARG30': 1, 'GLY31': -1}


Only one of the two can become an OpenFF `Molecule`.

In [4]:
from openff.toolkit import Molecule

molecule = Molecule.from_rdkit(protein.to_rdkit(), allow_undefined_stereo=True)
print("from Protein:", molecule.n_atoms, "atoms, net charge", molecule.total_charge)

try:
    Molecule.from_rdkit(generic.to_rdkit(), allow_undefined_stereo=True)
except Exception as error:
    print("from mb.load:", type(error).__name__, str(error).splitlines()[0])

from Protein: 470 atoms, net charge -1.0 elementary_charge
from mb.load: RuntimeError Pre-condition Violation
